# Phase 1: PTB-XL Data Loading, Preprocessing, and Inspection

This notebook demonstrates the foundational data pipeline for training 1D CNNs on the PTB-XL ECG dataset:
1. **Download PTB-XL (100 Hz version)** from PhysioNet.
2. **Parse `ptbxl_database.csv` & `scp_statements.csv`** and convert raw `scp_codes` to the 5 diagnostic superclasses (`NORM`, `MI`, `STTC`, `CD`, `HYP`).
3. **Load 12-lead signals with `wfdb.rdsamp`**, stack to `(N, 1000, 12)`, and transpose to `(N, 12, 1000)` for PyTorch.
4. **Standardize per lead** using the training set statistics (`strat_fold <= 8`).
5. **Plot 12-lead ECGs** to sanity-check raw signals and labels.
6. **Build PyTorch DataLoaders** with the patient-aware `strat_fold` split.

### Step 1: Install & Import Dependencies

In [ ]:
# If running in Google Colab, uncomment and run:
# !pip install -q wfdb torch pandas numpy scikit-learn matplotlib tqdm

import ast
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import torch
from tqdm.auto import tqdm

# If cloned as a repo, add root to sys.path so we can import src
current_dir = Path.cwd()
if (current_dir.parent / "src").exists():
    sys.path.append(str(current_dir.parent))
elif (current_dir / "src").exists():
    sys.path.append(str(current_dir))

from src.dataset import (
    SUPERCLASSES,
    STANDARD_LEAD_NAMES,
    load_ptbxl_metadata,
    load_raw_signals,
    standardize_per_lead,
    get_dataloaders,
    plot_12lead_ecg,
)

### Step 2: Download Dataset (if not already downloaded)

In Google Colab, the 100 Hz dataset archive can be downloaded and unzipped directly.

In [ ]:
DATA_DIR = Path("data/ptbxl")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# In Google Colab or Linux, run shell download for fastest transfer:
# !wget -nc -q -O ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
# !unzip -q -n ptbxl.zip -d data/ptbxl/
# Or if files are in data/ptbxl/ptb-xl-1.0.3, adjust DATA_DIR accordingly.

### Step 3: Load Metadata & Convert `scp_codes` to 5 Superclasses

In [ ]:
# Load metadata and map diagnostic statements
df, scp_df = load_ptbxl_metadata(data_dir=str(DATA_DIR))
df[['patient_id', 'age', 'sex', 'scp_codes', 'strat_fold'] + SUPERCLASSES].head()

### Step 4: Load Raw Signals & Transpose for PyTorch

In [ ]:
# Load 100 Hz signals: shape will be (N, 12, 1000)
X = load_raw_signals(df, data_dir=str(DATA_DIR), sampling_rate=100)
y = df[SUPERCLASSES].values.astype(np.float32)

print(f"Signals shape: {X.shape} (N, channels/leads, time_steps)")
print(f"Labels shape:  {y.shape} (N, 5 superclasses)")

### Step 5: Standardise Per Lead Using Training Set Statistics

We calculate the mean and standard deviation for each lead **strictly across folds 1–8** (`strat_fold <= 8`) to prevent data leakage into validation (fold 9) or test (fold 10).

In [ ]:
train_mask = (df.strat_fold <= 8).values
X_norm, lead_mean, lead_std = standardize_per_lead(X, train_mask)

print("Per-lead training means:\n", lead_mean.squeeze())
print("Per-lead training standard deviations:\n", lead_std.squeeze())
print(f"Normalized signals min: {X_norm.min():.2f}, max: {X_norm.max():.2f}")

### Step 6: Sanity-Check Visualizations (12-Lead ECGs)

In [ ]:
# Find a normal ECG sample (NORM)
norm_idx = np.where((df['NORM'] == 1) & (df[['MI', 'STTC', 'CD', 'HYP']].sum(axis=1) == 0))[0][0]
fig_norm = plot_12lead_ecg(
    X[norm_idx],
    title=f"Sample ECG #{df.index[norm_idx]} - Diagnostic: Normal (NORM)",
    save_path="sample_ecg_norm.png"
)
plt.show()

# Find a Myocardial Infarction sample (MI)
mi_idx = np.where(df['MI'] == 1)[0][0]
active_labels = [sc for sc in SUPERCLASSES if df[sc].iloc[mi_idx] == 1]
fig_mi = plot_12lead_ecg(
    X[mi_idx],
    title=f"Sample ECG #{df.index[mi_idx]} - Diagnostic: {', '.join(active_labels)}",
    save_path="sample_ecg_mi.png"
)
plt.show()

### Step 7: Build PyTorch DataLoaders & Verify Batch Dimensions

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(X_norm, y, df, batch_size=64)

# Inspect a single batch from train_loader
sample_batch_x, sample_batch_y = next(iter(train_loader))
print("Batch X shape:", sample_batch_x.shape, "-> (Batch Size, Leads, Time Steps)")
print("Batch y shape:", sample_batch_y.shape, "-> (Batch Size, 5 Classes)")
print("Sample label multi-hot vector:", sample_batch_y[0].numpy())